In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTENC
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from scikeras.wrappers import KerasClassifier
from sklearn.metrics import accuracy_score, classification_report


# --- Custom Transformers ---
# Handles missing values by filling them using related columns
class NullSatisfactionFiller(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        df = X.copy()
        df['Study/job Satisfaction'] = df['Study Satisfaction'].fillna(df['Job Satisfaction'])
        df['Academic/work Pressure'] = df['Academic Pressure'].fillna(df['Work Pressure'])
        return df

# Drops unnecessary columns like IDs, names, and unrelated attributes
class ColumnDropper(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        df = X.copy()
        cols_to_drop = ['CGPA','Name','Gender','id','City','Profession',
                        'Academic Pressure','Work Pressure','Degree',
                        'Study Satisfaction','Job Satisfaction']
        return df.drop(columns=cols_to_drop)

# Corrects categorical feature formats and converts categorical values
class FormatCorrector(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()
        # Dietary Habits
        correct_dh = ['Moderate', 'Unhealthy', 'Healthy']
        df['Dietary Habits'] = df['Dietary Habits'].apply(lambda x: x if x in correct_dh else None)
        
        # Sleep Duration
        correct_SD = ['Less than 5 hours', '1-2 hours', '2-3 hours', '3-4 hours', '4-5 hours',
                      '5-6 hours', '6-7 hours', '7-8 hours', '8-9 hours', '9-11 hours', 
                      '10-11 hours', 'More than 8 hours']
        sleep_map = {
            'Less than 5 hours': 4, '1-2 hours': 1.5, '2-3 hours': 2.5,
            '3-4 hours': 3.5, '4-5 hours': 4.5, '5-6 hours': 5.5, '6-7 hours': 6.5,
            '7-8 hours': 7.5, '8-9 hours': 8.5, '9-11 hours': 10,
            '10-11 hours': 10.5, 'More than 8 hours': 9
        }
        df['Sleep Duration'] = df['Sleep Duration'].apply(lambda x: x if x in correct_SD else np.nan)
        df['Sleep Duration'] = df['Sleep Duration'].map(sleep_map)
        return df

# Drops rows with missing values
class NullDropper(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.dropna().copy()

# Handles class imbalance by applying SMOTENC (Synthetic Minority Oversampling)
class SMOTENCTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, target='Depression'):
        self.target = target

    def fit(self, X, y=None):
        return self

    def transform(self, df):
        # Skip if target not present (i.e., in test data)
        if self.target not in df.columns:
            return df

        X_ = df.drop(columns=self.target)
        y_ = df[self.target]

        categorical_cols = [
            'Working Professional or Student',
            'Dietary Habits',
            'Have you ever had suicidal thoughts ?',
            'Financial Stress',
            'Family History of Mental Illness',
            'Academic/work Pressure',
            'Study/job Satisfaction'
        ]
        cat_idx = [X_.columns.get_loc(col) for col in categorical_cols if col in X_.columns]

        smote = SMOTENC(categorical_features=cat_idx, random_state=42)
        X_resampled, y_resampled = smote.fit_resample(X_, y_)

        # Create DataFrame with resampled data
        df_res = pd.DataFrame(X_resampled, columns=X_.columns)
        df_res[self.target] = y_resampled

        # Post-processing numeric columns
        for col in ['Age', 'Work/Study Hours', 'Study/job Satisfaction', 'Academic/work Pressure', 'Financial Stress']:
            if col in df_res.columns:
                df_res[col] = df_res[col].round().astype(int)

        # Apply valid range filters and ensure target is filtered too
        mask = (
            df_res['Age'].between(15, 60) &
            df_res['Work/Study Hours'].between(0, 12) &
            df_res['Study/job Satisfaction'].between(1, 5) &
            df_res['Academic/work Pressure'].between(1, 5) &
            df_res['Financial Stress'].between(1, 5)
        )
        df_res = df_res[mask].copy()
        y_resampled = y_resampled[mask]  # Filter target to match

        # Reattach filtered target to DataFrame
        df_res[self.target] = y_resampled

        return df_res

# Removes outliers in the Age column based on interquartile range (IQR)
class OutlierClipper(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, df):
        if 'Depression' not in df.columns:
            return df

        g1 = df[df['Depression'] == 1].copy()
        g0 = df[df['Depression'] == 0].copy()

        Q1 = g1['Age'].quantile(0.25)
        Q3 = g1['Age'].quantile(0.75)
        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        g1['Age'] = g1['Age'].clip(lower=lower, upper=upper)
        df_out = pd.concat([g0, g1]).reset_index(drop=True)
        
        # Verify sample count
        if len(df_out) != len(df):
            print(f"Warning: Sample count changed from {len(df)} to {len(df_out)} in OutlierClipper")
        
        return df_out
# Converts categorical variables into one-hot encoded format
class OneHotEncoderWrapper(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.categorical_cols = X.select_dtypes(include='object').columns.tolist()
        self.encoder = OneHotEncoder(drop='first', handle_unknown='ignore')
        self.encoder.fit(X[self.categorical_cols])
        return self

    def transform(self, X):
        df = X.copy()
        encoded = self.encoder.transform(df[self.categorical_cols]).toarray()
        encoded_df = pd.DataFrame(encoded, columns=self.encoder.get_feature_names_out(self.categorical_cols), index=df.index)
        df = pd.concat([df.drop(columns=self.categorical_cols), encoded_df], axis=1)
        return df

# Cleans column names by replacing spaces with underscores
class ColumnNameCleaner(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()
        df.columns = df.columns.str.replace(" ", "_")
        return df
    
# class ScalerWrapperm(BaseEstimator, TransformerMixin):
#     def fit(self, X, y=None):
#         self.scaler = StandardScaler()
#         self.scaler.fit(X)
#         return self
    
#     def transform(seelf, X):
#         return pd.DataFrame(self.scaler.transforum(X), columns=X.columns)

In [ ]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('null_satisfaction', NullSatisfactionFiller()),
    ('drop_columns', ColumnDropper()),
    ('correct_format', FormatCorrector()),
    ('drop_nulls', NullDropper()),
    ('smote', SMOTENCTransformer()),
    ('clip_outliers', OutlierClipper()),
    ('ohe', OneHotEncoderWrapper()),
    # ('scale', ScalerWrapper()),  
    ('clean_columns', ColumnNameCleaner())
])


In [3]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

train_df = pipeline.fit_transform(train_df)
test_df = pipeline.fit_transform(test_df)

In [4]:
# Define numeric columns to scale
numeric_cols = [
    'Age',
    'Sleep_Duration',
    'Work/Study_Hours',
    'Financial_Stress',
    'Study/job_Satisfaction',
    'Academic/work_Pressure'
]

# All others will be passed through
scaling_pipeline = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols)
], remainder='passthrough')

In [5]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 230122 entries, 0 to 230121
Data columns (total 12 columns):
 #   Column                                                Non-Null Count   Dtype  
---  ------                                                --------------   -----  
 0   Age                                                   230122 non-null  float64
 1   Sleep_Duration                                        230122 non-null  float64
 2   Work/Study_Hours                                      230122 non-null  int64  
 3   Financial_Stress                                      230122 non-null  int64  
 4   Study/job_Satisfaction                                230122 non-null  int64  
 5   Academic/work_Pressure                                230122 non-null  int64  
 6   Depression                                            230122 non-null  int64  
 7   Working_Professional_or_Student_Working_Professional  230122 non-null  float64
 8   Dietary_Habits_Moderate                     

In [6]:
from sklearn.model_selection import train_test_split

X = train_df.drop(columns='Depression')
y = train_df['Depression']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Verify split
print(f"Training Set: {X_train.shape}, {y_train.shape}")
print(f"Test Set: {X_val.shape}, {y_val.shape}")

Training Set: (184097, 11), (184097,)
Test Set: (46025, 11), (46025,)


In [10]:
# # 1️⃣ Build the Model
# def build_model(input_dim=11):  # Adjust input_dim to match feature count
#     model = Sequential([
#         Dense(64, activation='relu', input_shape=(input_dim,)),  # Input layer
#         Dropout(0.3),
#         Dense(32, activation='relu'),  # Hidden layer
#         Dropout(0.2),
#         Dense(1, activation='sigmoid')  # Output layer for binary classification
#     ])
#     # 2️⃣ Compile the Model
#     model.compile(optimizer=Adam(learning_rate=0.001), 
#                   loss='binary_crossentropy', 
#                   metrics=['accuracy'])
#     return model
class KerasNNClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, input_dim, epochs=50, batch_size=32, class_weight=None, verbose=0):
        self.input_dim = input_dim
        self.epochs = epochs
        self.batch_size = batch_size
        self.class_weight = class_weight
        self.verbose = verbose
        self.model_ = None  # Will hold the trained model

    def build_model(self):
        model = Sequential()
        # First hidden layer with 32 neurons and ReLU activation
        model.add(Dense(32, input_dim=self.input_dim, activation='relu'))
        model.add(Dropout(0.2))
        # Second hidden layer with 16 neurons
        model.add(Dense(16, activation='relu'))
        # Output layer for binary classification
        model.add(Dense(1, activation='sigmoid'))
        # Compile the model
        model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
        return model

    def fit(self, X, y):
        self.model_ = self.build_model()
        self.model_.fit(X, y,
                        epochs=self.epochs,
                        batch_size=self.batch_size,
                        class_weight=self.class_weight,
                        verbose=self.verbose)
        return self

    def predict(self, X):
        # Predict returns probabilities; we threshold them at 0.5 for binary classification
        preds = self.model_.predict(X)
        return (preds > 0.5).astype("int32").flatten()

# Create the pipeline with a StandardScaler and our custom neural network classifier
nn_pipeline = Pipeline([
    ('scaler', scaling_pipeline),
    ('nn', KerasNNClassifier(input_dim=X_train.shape[1],
                             epochs=20,
                             batch_size=32,
                             verbose=1))
])
# 3️⃣ Train the Model using Tensor Data
nn_pipeline.fit(X_train, y_train)



Epoch 1/20


f:\mental\myenv\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


5754/5754 ━━━━━━━━━━━━━━━━━━━━ 8s 1ms/step - accuracy: 0.9026 - loss: 0.2333
Epoch 2/20
5754/5754 ━━━━━━━━━━━━━━━━━━━━ 7s 1ms/step - accuracy: 0.9316 - loss: 0.1756
Epoch 3/20
5754/5754 ━━━━━━━━━━━━━━━━━━━━ 7s 1ms/step - accuracy: 0.9324 - loss: 0.1722
Epoch 4/20
5754/5754 ━━━━━━━━━━━━━━━━━━━━ 7s 1ms/step - accuracy: 0.9341 - loss: 0.1698
Epoch 5/20
5754/5754 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - accuracy: 0.9325 - loss: 0.1712
Epoch 6/20
5754/5754 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - accuracy: 0.9343 - loss: 0.1669
Epoch 7/20
5754/5754 ━━━━━━━━━━━━━━━━━━━━ 7s 1ms/step - accuracy: 0.9342 - loss: 0.1664
Epoch 8/20
5754/5754 ━━━━━━━━━━━━━━━━━━━━ 7s 1ms/step - accuracy: 0.9353 - loss: 0.1642
Epoch 9/20
5754/5754 ━━━━━━━━━━━━━━━━━━━━ 7s 1ms/step - accuracy: 0.9343 - loss: 0.1661
Epoch 10/20
5754/5754 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - accuracy: 0.9338 - loss: 0.1647
Epoch 11/20
5754/5754 ━━━━━━━━━━━━━━━━━━━━ 7s 1ms/step - accuracy: 0.9351 - loss: 0.1653
Epoch 12/20
5754/5754 ━━━━━━━━━━━━━━━━━━━

f:\mental\myenv\Lib\site-packages\sklearn\compose\_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('scaler',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['Age', 'Sleep_Duration',
                                                   'Work/Study_Hours',
                                                   'Financial_Stress',
                                                   'Study/job_Satisfaction',
                                                   'Academic/work_Pressure'])])),
                ('nn', KerasNNClassifier(epochs=20, input_dim=11, verbose=1))])

In [11]:
# Evaluate on the validation set
y_pred = nn_pipeline.predict(X_val)
print("Validation Accuracy:", accuracy_score(y_val, y_pred))
print("Classification Report:\n", classification_report(y_val, y_pred))

1439/1439 ━━━━━━━━━━━━━━━━━━━━ 1s 653us/step
Validation Accuracy: 0.935991309071157
Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.92      0.93     22873
           1       0.93      0.95      0.94     23152

    accuracy                           0.94     46025
   macro avg       0.94      0.94      0.94     46025
weighted avg       0.94      0.94      0.94     46025



In [17]:
import pandas as pd

# Get predictions
test_predictions = nn_pipeline.predict(test_df)

# Convert to DataFrame with column names
test_predictions_df = pd.DataFrame(test_predictions, columns=["Depression_Prediction"])

# Display sample predictions
print(test_predictions_df.head(10))


2930/2930 ━━━━━━━━━━━━━━━━━━━━ 2s 595us/step
   Depression_Prediction
0                      0
1                      0
2                      0
3                      1
4                      0
5                      0
6                      0
7                      0
8                      0
9                      1


In [16]:
import joblib

# Save the StandardScaler (from the pipeline)
joblib.dump(nn_pipeline.named_steps['scaler'], 'scaler.pkl')

# Save the trained Keras model (inside your custom classifier)
nn_pipeline.named_steps['nn'].model_.save('keras_model.h5')

print("Scaler and Keras model saved successfully.")

Scaler and Keras model saved successfully.
